In [ ]:
# === 노트북 공통 preamble (llm_math 패키지 로드) ===
import sys
from pathlib import Path

# llm_math 패키지를 찾을 수 있는 후보 경로들
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 현재 디렉토리의 상위 디렉토리들도 후보에 추가 (notebooks/ 폴더에서 실행 시)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math import 시도
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[주의] llm_math 패키지 로드 실패: {e}")
    print("  GitHub 레포를 클론하고 colab_setup.sh를 실행하세요.")
# === preamble 끝 ===


# Ch 11. RNN과 LSTM — 시퀀스를 다루는 최초의 접근

> **학습 목표**
> - RNN의 순환 구조를 수학으로 표현한다
> - 장기 의존성(long-term dependency) 문제와 기울기 소실을 이해한다
> - LSTM의 게이트 메커니즘이 왜 이를 완화하는지 안다

## 11.1 시퀀스 데이터와 가변 길이 문제

이미지는 고정 크기 벡터지만, 텍스트는 가변 길이 시퀀스. 어떻게 처리?

- **RNN**: 순환 구조로 시퀀스 처리
- **CNN (1D)**: 합성곱으로 지역 패턴
- **Transformer (Ch 14+)**: 어텐션으로 전역 의존성

이 장에서는 RNN/LSTM을 다룬다. 트랜스포머 이전의 표준이었다.

## 11.2 RNN의 수학적 구조

$$\mathbf{h}_t = \tanh(W_h \mathbf{h}_{t-1} + W_x \mathbf{x}_t + \mathbf{b})$$

- $\mathbf{x}_t \in \mathbb{R}^d$: 시간 $t$의 입력
- $\mathbf{h}_t \in \mathbb{R}^h$: 시간 $t$의 은닉 상태 (메모리)
- $W_h \in \mathbb{R}^{h \times h}$, $W_x \in \mathbb{R}^{h \times d}$: 파라미터

핵심: **같은 가중치를 매 시간 스텝에서 재사용** (파라미터 공유).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 간단한 RNN 셀 (NumPy)
class RNNCell:
    def __init__(self, input_dim, hidden_dim, seed=0):
        rng = np.random.default_rng(seed)
        self.Wx = rng.standard_normal((input_dim, hidden_dim)) * 0.1
        self.Wh = rng.standard_normal((hidden_dim, hidden_dim)) * 0.1
        self.b = np.zeros(hidden_dim)
        self.hidden_dim = hidden_dim

    def forward(self, x_seq):
        """x_seq: (T, input_dim) -> h_seq: (T, hidden_dim)"""
        T = x_seq.shape[0]
        h = np.zeros(self.hidden_dim)
        h_seq = []
        for t in range(T):
            h = np.tanh(x_seq[t] @ self.Wx + h @ self.Wh + self.b)
            h_seq.append(h.copy())
        return np.array(h_seq)

    def last_hidden(self, x_seq):
        return self.forward(x_seq)[-1]

# Test
rnn = RNNCell(input_dim=4, hidden_dim=8)
x_seq = np.random.randn(10, 4)  # Length 10 시퀀스
h_seq = rnn.forward(x_seq)
print(f"입력 시퀀스: {x_seq.shape}")
print(f"은닉 상태 시퀀스: {h_seq.shape}")
print(f"마지막 은닉 상태: {h_seq[-1].round(4)}")


## 11.3 BPTT (Backpropagation Through Time)

RNN의 역전파 = 시간 축으로 펼친 계산 그래프에서 역전파.

$$\frac{\partial \mathcal{L}}{\partial \mathbf{h}_0} = \prod_{t=1}^{T} \frac{\partial \mathbf{h}_t}{\partial \mathbf{h}_{t-1}} \cdot \frac{\partial \mathcal{L}}{\partial \mathbf{h}_T}$$

$\frac{\partial \mathbf{h}_t}{\partial \mathbf{h}_{t-1}} = \mathrm{diag}(1 - \mathbf{h}_t^2) W_h$ (tanh의 도함수 포함).

문제: $\mathrm{diag}(1 - \mathbf{h}_t^2) \leq 1$이고 $W_h$의 고윳값 절댓값이 1보다 작으면 **그래디언트 소실**, 크면 **폭발**.


In [ ]:
# 그래디언트 소실/폭발 시뮬레이션
def simulate_rnn_gradient(T, Wh_norm, seed=0):
    """T Step RNN에서 h_0에 대한 Gradient Magnitude."""
    rng = np.random.default_rng(seed)
    h = rng.standard_normal(8)
    grad = np.ones(8)  # dL/dh_T
    for t in range(T):
        # Wh가 단위 Matrix의 norm배
        Wh = np.eye(8) * Wh_norm
        h_prev = rng.standard_normal(8)
        h = np.tanh(h_prev @ Wh)
        # Backward Pass: grad = grad @ Wh @ diag(1 - h^2)
        diag = 1 - h**2
        grad = grad @ Wh * diag
    return np.linalg.norm(grad)

print("RNN 그래디언트 소실/폭발 시뮬레이션:")
print(f"{'T':>5} | {'||Wh||=0.5 (소실)':>20} | {'||Wh||=1.0':>15} | {'||Wh||=1.5 (폭발)':>20}")
print("-" * 65)
for T in [5, 10, 20, 50, 100]:
    g_small = simulate_rnn_gradient(T, 0.5)
    g_one = simulate_rnn_gradient(T, 1.0)
    g_big = simulate_rnn_gradient(T, 1.5)
    print(f"{T:>5} | {g_small:>20.4e} | {g_one:>15.4e} | {g_big:>20.4e}")

print("\n=> ||Wh||<1이Plane Gradient가 지수적 Decrease (소실)")
print("=> ||Wh||>1이Plane Gradient가 지수적 Increase (폭발)")


## 11.4 LSTM — 게이트 메커니즘

LSTM (1997, Hochreiter & Schmidhuber)은 게이트로 그래디언트 흐름 제어.

$$\mathbf{f}_t = \sigma(W_f [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_f) \quad \text{(forget gate)}$$
$$\mathbf{i}_t = \sigma(W_i [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_i) \quad \text{(input gate)}$$
$$\tilde{\mathbf{c}}_t = \tanh(W_c [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_c) \quad \text{(candidate)}$$
$$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t \quad \text{(cell state)}$$
$$\mathbf{o}_t = \sigma(W_o [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_o) \quad \text{(output gate)}$$
$$\mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t) \quad \text{(hidden state)}$$

핵심: **cell state $\mathbf{c}_t$는 덧셈으로만 갱신** → 그래디언트가 잘 흐름.


In [ ]:
# LSTM 셀 구현 (NumPy)
class LSTMCell:
    def __init__(self, input_dim, hidden_dim, seed=0):
        rng = np.random.default_rng(seed)
        # 4개 게이트: forget, input, candidate, output
        self.W = rng.standard_normal((input_dim + hidden_dim, 4 * hidden_dim)) * 0.1
        self.b = np.zeros(4 * hidden_dim)
        self.hidden_dim = hidden_dim

    def forward(self, x_seq):
        T = x_seq.shape[0]
        h = np.zeros(self.hidden_dim)
        c = np.zeros(self.hidden_dim)
        h_seq, c_seq = [], []
        for t in range(T):
            # 결합: [h, x]
            hx = np.concatenate([h, x_seq[t]])
            gates = hx @ self.W + self.b
            f, i, g, o = np.split(gates, 4)
            f = 1 / (1 + np.exp(-f))  # sigmoid
            i = 1 / (1 + np.exp(-i))
            g = np.tanh(g)
            o = 1 / (1 + np.exp(-o))
            c = f * c + i * g  # cell state (덧셈 갱신!)
            h = o * np.tanh(c)
            h_seq.append(h.copy()); c_seq.append(c.copy())
        return np.array(h_seq), np.array(c_seq)

# 테스트
lstm = LSTMCell(input_dim=4, hidden_dim=8)
h_seq, c_seq = lstm.forward(x_seq)
print(f"LSTM Output: h_seq {h_seq.shape}, c_seq {c_seq.shape}")
print(f"마지막 h: {h_seq[-1].round(4)}")
print(f"마지막 c: {c_seq[-1].round(4)}")


## 11.5 문자 단위 언어 모델

간단한 문자 단위 RNN/LSTM으로 텍스트를 생성해 보자.


In [ ]:
# 문자 단위 LSTM 언어 모델 (PyTorch)
import torch
import torch.nn as nn

# Tiny Shakespeare 작은 샘플
from llm_math.data import load_tiny_shakespeare
text = load_tiny_shakespeare(max_chars=20000)
chars = sorted(set(text))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}
vocab_size = len(chars)
print(f"텍스트 Length: {len(text)}, Vocabulary Size: {vocab_size}")
print(f"문자: {chars[:20]}...")

# 데이터 준비
seq_len = 50
data = np.array([char_to_idx[c] for c in text])

def make_batch(data, seq_len, batch_size=32):
    """랜덤하게 Batch Generation."""
    n = len(data) - seq_len - 1
    starts = np.random.randint(0, n, batch_size)
    X = np.array([data[s:s+seq_len] for s in starts])
    y = np.array([data[s+1:s+1+seq_len] for s in starts])
    return torch.tensor(X, dtype=torch.long), torch.tensor(y, dtype=torch.long)

# Model
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, n_layers=1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, n_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        emb = self.embed(x)  # (B, T, E)
        out, hidden = self.lstm(emb, hidden)  # (B, T, H)
        logits = self.fc(out)  # (B, T, V)
        return logits, hidden

model = CharLSTM(vocab_size, embed_dim=32, hidden_dim=64)
n_params = sum(p.numel() for p in model.parameters())
print(f"\n모델 파라미터 수: {n_params:,}")


In [ ]:
# 학습
import time

loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.003)

n_steps = 300
losses = []
t0 = time.time()
for step in range(n_steps):
    X, y = make_batch(data, seq_len, batch_size=32)
    opt.zero_grad()
    logits, _ = model(X)
    # (B*T, V) vs (B*T,)
    loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
    opt.step()
    losses.append(loss.item())
    if (step + 1) % 50 == 0:
        print(f"Step {step+1}: loss={loss.item():.4f}")
print(f"\nTraining Time: {time.time() - t0:.1f}초")


In [ ]:
# 텍스트 생성
def generate(model, seed_text, length=200, temperature=0.8):
    model.eval()
    chars_idx = [char_to_idx[c] for c in seed_text]
    hidden = None
    with torch.no_grad():
        x = torch.tensor([chars_idx], dtype=torch.long)
        for _ in range(length):
            logits, hidden = model(x, hidden)
            # 마지막 시간 스텝
            logit = logits[0, -1] / temperature
            probs = torch.softmax(logit, dim=0)
            next_idx = torch.multinomial(probs, 1).item()
            chars_idx.append(next_idx)
            # 다음 입력은 마지막 문자만
            x = torch.tensor([[next_idx]], dtype=torch.long)
    return ''.join(idx_to_char[i] for i in chars_idx)

print("=== Generation된 텍스트 (Training 후) ===\n")
print(generate(model, "First Citizen:\n", length=300))


In [ ]:
# 학습 곡선
plt.figure(figsize=(10, 4))
plt.plot(losses, alpha=0.3, color='blue')
# 이동 Mean
window = 20
smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
plt.plot(range(window-1, len(losses)), smoothed, 'r-', linewidth=2, label='이동Mean')
plt.xlabel('Step'); plt.ylabel('Loss')
plt.title('Character-level LSTM Learning Curve')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch11_char_lstm.png', dpi=100, bbox_inches='tight')
plt.show()


## 11.6 GRU — LSTM의 단순화

GRU (2014, Cho et al.)는 게이트 2개로 단순화:

$$\mathbf{z}_t = \sigma(W_z [\mathbf{h}_{t-1}, \mathbf{x}_t]) \quad \text{(update gate)}$$
$$\mathbf{r}_t = \sigma(W_r [\mathbf{h}_{t-1}, \mathbf{x}_t]) \quad \text{(reset gate)}$$
$$\tilde{\mathbf{h}}_t = \tanh(W [\mathbf{r}_t \odot \mathbf{h}_{t-1}, \mathbf{x}_t])$$
$$\mathbf{h}_t = (1 - \mathbf{z}_t) \odot \mathbf{h}_{t-1} + \mathbf{z}_t \odot \tilde{\mathbf{h}}_t$$

cell state가 없이 더 단순. 성능은 LSTM과 비슷.


## 11.7 어텐션의 등장 배경 — RNN의 병목

RNN의 한계:
1. **순차 처리**: 병렬화 불가 → 느림
2. **장기 의존성**: LSTM도 먼 정보는 잊음
3. **마지막 은닉 상태 병목**: 인코더의 모든 정보가 하나의 벡터에 압축

**어텐션(attention)**은 디코더가 매 스텝마다 인코더의 모든 은닉 상태를 참조하게 해 이 병목을 해결. (Bahdanau et al., 2014)

이것이 트랜스포머(Ch 14)의 시초가 된다.

## 11.8 핵심 정리

| 개념 | 수식 | 특징 |
|---|---|---|
| RNN | $\mathbf{h}_t = \tanh(W_h \mathbf{h}_{t-1} + W_x \mathbf{x}_t)$ | 순차, 소실/폭발 |
| LSTM | 게이트 + cell state (덧셈) | 장기 의존성 해결 |
| GRU | 게이트 2개 (단순화) | LSTM과 비슷 |
| BPTT | 시간축 역전파 | $\prod \frac{\partial h_t}{\partial h_{t-1}}$ |

## 연습문제

1. RNN으로 "hello world"를 문자 단위로 학습시키고 생성하라.
2. LSTM과 GRU를 같은 데이터로 학습하고 수렴 속도를 비교하라.
3. 시퀀스 길이 10, 50, 100에 대해 RNN과 LSTM의 그래디언트 크기를 비교하라.
4. RNN이 긴 시퀀스에서 첫 부분을 잊는 현상을 실험으로 보이라.
5. 문자 단위 LSTM의 temperature를 0.3, 0.8, 1.5로 바꿔가며 생성 텍스트를 비교하라.

> 해설: `solutions/ch11_solutions.ipynb`
